<a href="https://colab.research.google.com/github/RuiRodrigues-lab/DataScienceFE/blob/Locker/CP4_2(TH).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.diagnostic import lilliefors
from scipy.stats import wilcoxon
import rpy2.robjects as ro
from scipy.stats import ttest_ind
#Precisamos desta biblioteca para podermos escolher um ficheiro local
#Se o ficheiro vier por API ou tivermos um link, é so alterar a forma de import
from google.colab import files

# 1️⃣ Faz upload do ficheiro (vai abrir uma janela para escolher no teu PC)
uploaded = files.upload()

# 2️⃣ Guarda o nome do ficheiro (Colab mostra o nome depois do upload)
filename = list(uploaded.keys())[0]

# 3️⃣ Lê o Excel, por default lê sempre a primeira tab, por isso podemos usar o "sheet_name"
# Se tivermos dados em varias tabs, devemos usar uma Dataframe(df) para cada uma das tabs
df = pd.read_excel(filename, sheet_name='Exerc2')
df.head()

Saving CP4.xlsx to CP4 (1).xlsx


,Tempo,Departamento
0,12.0,B
1,13.5,B
2,11.7,B
3,12.6,B
4,11.4,B


In [3]:
n_total = len(df["Tempo"])
n_non_missing = df["Tempo"].notna().sum()

print("Total:", n_total)
print("Não-missing:", n_non_missing)

Total: 99
Não-missing: 99


In [9]:
# 2) Garantir que Departamento é categórico com níveis A e B
df["Departamento"] = pd.Categorical(df["Departamento"], categories=["A", "B"])

# 3) Criar grupos A e B (como grupoA e grupoB no R)
grupoA = df.loc[df["Departamento"] == "A", "Tempo"].dropna()
grupoB = df.loc[df["Departamento"] == "B", "Tempo"].dropna()

print("\nlength(grupoA) =", len(grupoA))
print("length(grupoB) =", len(grupoB))


length(grupoA) = 48
length(grupoB) = 51


In [10]:
# 3) Criar grupos A e B (como grupoA e grupoB no R)
grupoA = df.loc[df["Departamento"] == "A", "Tempo"].dropna()
grupoB = df.loc[df["Departamento"] == "B", "Tempo"].dropna()

print("\nlength(grupoA) =", len(grupoA))
print("length(grupoB) =", len(grupoB))

# 4) Teste de normalidade Lilliefors (Kolmogorov-Smirnov com correcção) para cada grupo
D_A, p_A = lilliefors(grupoA, dist='norm')
D_B, p_B = lilliefors(grupoB, dist='norm')

print("\nLilliefors (grupo A):")
print("D =", D_A)
print("p-value =", p_A)

print("\nLilliefors (grupo B):")
print("D =", D_B)
print("p-value =", p_B)


length(grupoA) = 48
length(grupoB) = 51

Lilliefors (grupo A):
D = 0.13356932632459811
p-value = 0.03337517610695484

Lilliefors (grupo B):
D = 0.10527177397482711
p-value = 0.1817010570715346


In [11]:


# Welch bilateral (equal_var=False)
t_stat, p_two_sided = ttest_ind(grupoA, grupoB, equal_var=False)

# Converter para unilateral: H1: mu_A < mu_B
if t_stat < 0:
    p_value = p_two_sided / 2
else:
    p_value = 1 - p_two_sided / 2

print("Welch Two Sample t-test (A < B)")
print("t =", t_stat)
print("p-value =", p_value)

Welch Two Sample t-test (A < B)
t = -11.702050736805548
p-value = 1.281080607012368e-19


In [12]:
# grupos A e B já definidos
grupoA = df.loc[df["Departamento"] == "A", "Tempo"].dropna()
grupoB = df.loc[df["Departamento"] == "B", "Tempo"].dropna()

print("mean in group A =", grupoA.mean())
print("mean in group B =", grupoB.mean())

print("length(grupoA) =", len(grupoA))
print("length(grupoB) =", len(grupoB))

print("sd(grupoA) =", grupoA.std(ddof=1))
print("sd(grupoB) =", grupoB.std(ddof=1))

#Foram selecionados tempos de experiência de 48 colaboradores do departamento A e 51 do
#departamento B. Relativamente aos dados recolhidos, verifica-se que, o tempo médio de experiência
#dos colaboradores do departamento A (11,18125 anos) é inferior ao tempo médio de experiência dos
#colaboradores do departamento B (12,61176 anos). A dispersão dos tempos de experiência
#(relativamente à média) dos colaboradores do departamento A (0,4536642 anos) é inferior à dos
#colaboradores do departamento B (0,7371966 anos).

#p-value< 2,2x10-160→ rejeitar H0
#Para qualquer nível de significância (0,05 ou 0,01 ou 0,1), existe evidência estatística para rejeitar H0,
#ou seja, o tempo médio de experiência dos colaboradores do departamento A é significativamente
#inferior ao tempo médio de experiência dos colaboradores do departamento B

mean in group A = 11.18125
mean in group B = 12.61176470588235
length(grupoA) = 48
length(grupoB) = 51
sd(grupoA) = 0.4536641601589944
sd(grupoB) = 0.73719659761112
